# Buổi 3.10: Fine-tuning đầy đủ và PEFT
Có cần cập nhật tất cả weights của pretrained model không?

## Kiến thức
- Full fine-tune cập nhật base; frozen probe chỉ cập nhật head.
- LoRA học update low-rank; adapter phải lưu cùng đúng base revision.
- Đếm trainable params, đo validation và inference cost khi so phương án.

**Công thức:** ΔW=B×A, params=r(in+out)


In [1]:
# -*- coding: utf-8 -*-
import torch
from torch import nn
torch.manual_seed(42)

# Minh họa cơ chế LoRA ở một linear layer nhỏ, không giả là fine-tune LLM.
d,r = 4,2
base = nn.Linear(d,d,bias=False)
for p in base.parameters(): p.requires_grad=False
A = nn.Parameter(torch.randn(r,d)*.01)
B = nn.Parameter(torch.zeros(d,r))
X = torch.ones(3,d)
output = base(X)+X@A.T@B.T
loss = output.square().mean(); loss.backward()
print("Base params:",sum(p.numel() for p in base.parameters()))
print("Adapter params:",A.numel()+B.numel())
print("Base grad:",base.weight.grad)
print("B có gradient:",B.grad is not None)
assert base.weight.grad is None and A.numel()+B.numel()==16


Base params: 16
Adapter params: 16
Base grad: None
B có gradient: True


## Đọc kết quả
Model/adapter và inference theo chương; thực hành cần GPU như nguồn.
## Nguồn
AIO 58.1–58.2 và59.2.
AIO 58.3 trắc nghiệm;59.2 PEFT for Multiple-choice QA và59.3 trắc nghiệm.


## Hai bài trên lớp
1. Tổng số parameters trainable của các layers được đánh trainable=true.
Input: `{"layers": [{"name": "base", "count": 16, "trainable": false}, {"name": "head", "count": 9, "trainable": true}]}`
2. Adapter A: r x in, B: out x r; trả adapter params và base params(out*in).
Input: `{"in": 4, "out": 4, "r": 2}`

Viết cell mới để làm. Đối chiếu trong BAI_HOC.html sau khi thử.
## BTVN
- BTVN 1: BTVN_01.txt
- BTVN 2: BTVN_02.txt
- BTVN 3: BTVN_03.txt
